In [50]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

In [2]:
df = pd.read_csv('/content/drive/MyDrive/Fraud_detection_system/onlinefraud.csv')

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
df['type'].unique()

array(['PAYMENT', 'TRANSFER', 'CASH_OUT', 'DEBIT', 'CASH_IN'],
      dtype=object)

In [5]:
df = df.drop('isFlaggedFraud',axis=1)

In [6]:
df = df.rename(columns={'step':'hours'})

In [7]:
df.drop(['nameOrig', 'nameDest'], axis=1, inplace=True)

In [8]:
df = pd.get_dummies(df, columns=['type'], drop_first=True,dtype='int')

In [9]:
df.head()

,hours,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,0,1,0


In [10]:
# df['orig_balance_diff'] = df['oldbalanceOrg'] - df['newbalanceOrig']
# df['dest_balance_diff'] = df['newbalanceDest'] - df['oldbalanceDest']

In [11]:
# df['orig_error'] = df['amount'] - df['orig_balance_diff']
# df['dest_error'] = df['amount'] - df['dest_balance_diff']

In [12]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [14]:
model = RandomForestClassifier( class_weight='balanced',random_state=42)

result = model.fit(X_train, y_train)

In [15]:
ytrain_pred = result.predict(X_train)

In [ ]:
print(confusion_matrix(y_train, ytrain_pred))

In [17]:
print(classification_report(y_train, ytrain_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   4448056
           1       1.00      1.00      1.00      5778

    accuracy                           1.00   4453834
   macro avg       1.00      1.00      1.00   4453834
weighted avg       1.00      1.00      1.00   4453834



In [18]:
feature_importance = pd.Series(
    result.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance.head(10))

oldbalanceOrg     0.292131
amount            0.172362
newbalanceOrig    0.156517
type_TRANSFER     0.093867
hours             0.066199
type_CASH_OUT     0.065362
newbalanceDest    0.059850
type_PAYMENT      0.049382
oldbalanceDest    0.044180
type_DEBIT        0.000149
dtype: float64


In [19]:
ytest_pred = result.predict(X_test)
ytest_prob = result.predict_proba(X_test)[:, 1]

In [26]:
accuracy_score(y_test,ytest_pred)

0.9996882835477628

In [27]:
confusion_matrix(y_test, ytest_pred)

array([[1906314,      37],
       [    558,    1877]])

In [30]:
print(classification_report(y_test, ytest_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906351
           1       0.98      0.77      0.86      2435

    accuracy                           1.00   1908786
   macro avg       0.99      0.89      0.93   1908786
weighted avg       1.00      1.00      1.00   1908786



In [29]:
"ROC-AUC Score:", roc_auc_score(y_test, ytest_prob)

('ROC-AUC Score:', np.float64(0.9930938057104156))

In [51]:
# Save model
pickle.dump(model, open("model.pkl", "wb"))

# Save column order
pickle.dump(X.columns, open("columns.pkl", "wb"))

In [57]:
!pip install streamlit -q
!wget -q -O - ipv4.icanhazip.com

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 46.6 MB/s eta 0:00:00
136.110.60.13


In [61]:
! streamlit run /content/drive/MyDrive/Fraud_detection_system/app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.110.60.13:8501

your url is: https://giant-items-stand.loca.lt
  Stopping...
^C
